## In this Notebook, we are gonna focus on extracting action items in JSON Fomrat so that it can be rendered on frontend

In [1]:
summary = """
**Meeting Summary: Business Analysis Discussion**

**Date:** 2025-12-12
**Time:** 10:00 AM - 10:45 AM
**Participants:** Sarah (PM) and Ravi (BA)

**Key Discussion Points:**

1. Review of initial findings from the analysis on the new feature rollout, including:
        * Adoption rate of the new recommendation engine (22% among active users)
        * Correlation between subscription tier and engagement with the new feature
        * Prime usage windows (11 AM and 3 PM daily)
2. Analysis of friction points and potential areas for improvement:
        * Users tend to drop off when navigating from the home page to the recommendations tab
        * Hesitation around filtering options due to potential lack of intuitiveness
        * Load times spike slightly when multiple filters are applied
3. Cohort analysis indicating users who engage with at least three personalized recommendations per week have a retention rate 18% higher than those who don't
4. Recommendations for next steps:
        * Redesign the filtering UI to reduce friction
        * Optimize backend queries to improve load time
        * Consider an onboarding prompt highlighting the top three recommendations for new users

**Decisions Made:**

1. UX and performance optimizations are critical before a broader rollout of the new feature.
2. Prioritize UI redesign and backend optimization for the next sprint.

**Action Items:**

1. Further analysis on the new recommendation engine adoption rate and its correlation with subscription tier.
2. Consideration of prime usage windows for future feature rollouts.
3. Draft a detailed plan for UI redesign and backend optimization, including estimated effort and dependencies, to be completed by Ravi by tomorrow afternoon.
4. Prioritize UX and performance optimizations to address friction points and improve user experience.

**Important Insights:**

1. The new recommendation engine has an adoption rate of 22% among active users.
2. Premium subscribers have a higher adoption rate (35%) compared to non-premium subscribers.
3. There is a clear correlation between subscription tier and engagement with the new feature.
4. Users who engage with at least three personalized recommendations per week have a retention rate 18% higher than those who don't.

**Next Steps:**

Ravi will continue analyzing the data to provide more insights and draft a detailed plan for UI redesign and backend optimization. Sarah will consider the findings when planning future feature rollouts, prioritizing UX and performance optimizations to address friction points and improve user experience."""

> Used Generated Summary for demonstration for identifying and extracting Action Items

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_groq import ChatGroq
from typing import List, Optional
from pydantic import BaseModel
from dotenv import load_dotenv, find_dotenv
import os
load_dotenv(find_dotenv())

groq_apikey = os.getenv("GROQ_API_KEY")

class ActionItemObject(BaseModel):
    action: str
    assignee: Optional[str] = None
    deadline: Optional[str] = None
    feedback: Optional[str] = None
    
class ActionItems(BaseModel):
    items: List[ActionItemObject]
    
parser = PydanticOutputParser(pydantic_object=ActionItems)
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"$defs": {"ActionItemObject": {"properties": {"action": {"title": "Action", "type": "string"}, "assignee": {"anyOf": [{"type": "string"}, {"type": "null"}], "default": null, "title": "Assignee"}, "deadline": {"anyOf": [{"type": "string"}, {"type": "null"}], "default": null, "title": "Deadline"}, "feedback": {"anyOf": [{"type": "string"}, {"type": "null"}], "default": null, "title": "Feedback"}}, "required": ["action"], "title": "ActionItemObject", "type": "object"}}, "properties": {"items": {"items": {"$ref": "#/$defs/ActionItemObject"}, "tit

In [ ]:
prompt_action_items = f"""You are an meeting analyst, Given the generated summary by a large language model, your work is to extract action items in JSON format as schema specified as under:

Schema Instructions:
{parser.get_format_instructions()}

Summary:
{summary}
"""
llm = ChatGroq(
    api_key=groq_apikey,
    model="llama-3.3-70b-versatile", # llama-3.3-70b-versatile good for summarizing texts and providing good context
    temperature=0.0,
    max_retries=2
)
